In [2]:
!git clone https://github.com/NhanElastic/Canteen_Rush.git

%cd Canteen_Rush
!git switch AI


Cloning into 'Canteen_Rush'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 30 (delta 7), reused 29 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 9.14 KiB | 9.14 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Canteen_Rush/Canteen_Rush
branch 'AI' set up to track 'origin/AI'.
Switched to a new branch 'AI'


In [5]:
%cd training/
!pip install -r requirements.txt

[Errno 2] No such file or directory: 'training/'
/content/Canteen_Rush/training
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 43.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 144.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [onnxscript]


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import io
import zipfile
import urllib.request
from pathlib import Path
import numpy as np
import soundfile as sf
from datasets import Audio, load_dataset

RAW_DIR = Path("raw_collected")
UNK_DIR = RAW_DIR / "unknown"
SAMPLE_RATE = 16000
CHUNK_SAMPLES = int(SAMPLE_RATE * 1.5)

# 1. Giải nén 10 lớp món ăn từ Google Drive (chỉnh lại đường dẫn nếu bạn lưu chỗ khác)
drive_zip = Path("/content/drive/MyDrive/canteen_ai/raw_collected.zip")
if drive_zip.exists():
    with zipfile.ZipFile(drive_zip, "r") as zf:
        zf.extractall(RAW_DIR)
    print("Đã giải nén dữ liệu 10 món ăn từ Google Drive.")
else:
    print(f"[Lưu ý] Chưa thấy {drive_zip}, hãy kiểm tra lại đường dẫn file zip trên Drive.")

# 2. Tải 40 mẫu tiếng ồn căng-tin từ DEMAND (Zenodo)
UNK_DIR.mkdir(parents=True, exist_ok=True)
print("Đang tải tiếng ồn căng-tin DEMAND (~107 MB)...")
zip_path, _ = urllib.request.urlretrieve("https://zenodo.org/records/1227121/files/PCAFETER_16k.zip")
with zipfile.ZipFile(zip_path, "r") as zf:
    wav, _ = sf.read(io.BytesIO(zf.read("PCAFETER/ch01.wav")), dtype="float32")

caf_count = 0
for start in range(0, len(wav) - CHUNK_SAMPLES + 1, CHUNK_SAMPLES):
    if caf_count >= 40:
        break
    caf_count += 1
    sf.write(UNK_DIR / f"unk_cafeteria_{caf_count:03d}.wav", wav[start : start + CHUNK_SAMPLES], SAMPLE_RATE)

# 3. Tải 40 mẫu giọng nói tiếng Việt ngoài thực đơn từ Hugging Face (LSVSC)
print("Đang tải 40 mẫu giọng nói tiếng Việt nền từ LSVSC...")
ds = load_dataset("doof-ferb/LSVSC", split="train", streaming=True)
ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

vi_count = 0
for row in ds:
    arr = np.asarray(row["audio"]["array"], dtype=np.float32)
    if len(arr) < int(SAMPLE_RATE * 0.5):
        continue
    vi_count += 1
    sf.write(UNK_DIR / f"unk_vispeech_{vi_count:03d}.wav", arr[:CHUNK_SAMPLES], SAMPLE_RATE)
    if vi_count >= 40:
        break

print(f"\nTổng hợp dữ liệu trong '{RAW_DIR}':")
for d in sorted(RAW_DIR.iterdir()):
    if d.is_dir():
        print(f"  - {d.name:<12}: {len(list(d.iterdir()))} files")

[Lưu ý] Chưa thấy /content/drive/MyDrive/canteen_ai/raw_collected.zip, hãy kiểm tra lại đường dẫn file zip trên Drive.
Đang tải tiếng ồn căng-tin DEMAND (~107 MB)...
Đang tải 40 mẫu giọng nói tiếng Việt nền từ LSVSC...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]


Tổng hợp dữ liệu trong 'raw_collected':
  - unknown     : 80 files


In [27]:
import asyncio
import random
from pathlib import Path
import edge_tts

RAW_DIR = Path("raw_collected")

# 1. Xóa toàn bộ file âm thanh rỗng/lỗi (< 500 bytes) từ lần chạy bị ngắt trước đó
removed = 0
for f in RAW_DIR.rglob("*"):
    if f.is_file() and f.stat().st_size < 500:
        f.unlink()
        removed += 1
print(f"Đã dọn dẹp {removed} file lỗi/rỗng.")

MENU_PHRASES = {
    "pho": ["Phở", "Cho một phở", "Phở bò"],
    "banh_mi": ["Bánh mì", "Một bánh mì", "Ổ bánh mì"],
    "com_tam": ["Cơm tấm", "Một cơm tấm", "Dĩa cơm tấm"],
    "bun_cha": ["Bún chả", "Một bún chả", "Suất bún chả"],
    "goi_cuon": ["Gỏi cuốn", "Hai gỏi cuốn", "Một gỏi cuốn"],
    "nem_ran": ["Nem rán", "Một nem rán", "Phần nem rán"],
    "coca": ["Coca", "Lon Coca", "Cho một Coca"],
    "pepsi": ["Pepsi", "Lon Pepsi", "Cho một Pepsi"],
    "tra_da": ["Trà đá", "Ly trà đá", "Cho một trà đá"],
    "mien_ga": ["Miến gà", "Một miến gà", "Tô miến gà"],
}

VOICES = ["vi-VN-HoaiMyNeural", "vi-VN-NamMinhNeural"]
RATES = ["-10%", "-5%", "+0%", "+8%", "+15%"]
PITCHES = ["-4Hz", "+0Hz", "+4Hz"]

async def gen_with_retry(sem: asyncio.Semaphore, text: str, voice: str, rate: str, pitch: str, out_file: Path):
    if out_file.exists() and out_file.stat().st_size >= 500:
        return  # Bỏ qua file đã tạo thành công ở lần trước

    async with sem:
        for attempt in range(4):
            try:
                comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
                await comm.save(str(out_file))
                if out_file.exists() and out_file.stat().st_size >= 500:
                    await asyncio.sleep(0.15)
                    return
            except Exception:
                if out_file.exists():
                    out_file.unlink()
                await asyncio.sleep(1.0 * (attempt + 1))

async def complete_mock_menu():
    sem = asyncio.Semaphore(2)  # Giới hạn 2 luồng để tránh bị Edge TTS ngắt kết nối
    tasks = []
    for cls_name, texts in MENU_PHRASES.items():
        cls_dir = RAW_DIR / cls_name
        cls_dir.mkdir(parents=True, exist_ok=True)
        idx = 1
        for voice in VOICES:
            for text in texts:
                for rate in RATES:
                    out_file = cls_dir / f"tts_{idx:03d}.mp3"
                    tasks.append(gen_with_retry(sem, text, voice, rate, random.choice(PITCHES), out_file))
                    idx += 1
    await asyncio.gather(*tasks)

    print("\nThống kê số file hợp lệ hiện tại:")
    for d in sorted(RAW_DIR.iterdir()):
        if d.is_dir():
            valid_files = [f for f in d.iterdir() if f.is_file() and f.stat().st_size >= 500]
            print(f"  - {d.name:<12}: {len(valid_files)} files")

await complete_mock_menu()

Đã dọn dẹp 11 file lỗi/rỗng.

Thống kê số file hợp lệ hiện tại:
  - banh_mi     : 30 files
  - bun_cha     : 30 files
  - coca        : 30 files
  - com_tam     : 30 files
  - goi_cuon    : 30 files
  - mien_ga     : 30 files
  - nem_ran     : 30 files
  - pepsi       : 30 files
  - pho         : 30 files
  - tra_da      : 30 files
  - unknown     : 80 files


In [29]:
!python train.py


Lớp (Class)        |  Tổng | Train (80%) | Val (20%)
------------------------------------------------------
banh_mi            |    30 |          24 |         6
bun_cha            |    30 |          24 |         6
coca               |    30 |          24 |         6
com_tam            |    30 |          24 |         6
goi_cuon           |    30 |          24 |         6
mien_ga            |    30 |          24 |         6
nem_ran            |    30 |          24 |         6
pepsi              |    30 |          24 |         6
pho                |    30 |          24 |         6
tra_da             |    30 |          24 |         6
unknown            |    80 |          64 |        16
------------------------------------------------------
Tổng cộng (Total)  |   380 |         304 |        76
Phát hiện 11 lớp: ['banh_mi', 'bun_cha', 'coca', 'com_tam', 'goi_cuon', 'mien_ga', 'nem_ran', 'pepsi', 'pho', 'tra_da', 'unknown']
Số mẫu hợp lệ -> Train: 304 | Val: 76
Thiết bị huấn luyện: cuda
Epoch 

In [30]:
!python verify_onnx.py

Lớp thực tế        | Đúng / Tổng | Độ chính xác
------------------------------------------------
banh_mi            |    6 / 6    |      100.0%
bun_cha            |    6 / 6    |      100.0%
coca               |    6 / 6    |      100.0%
com_tam            |    6 / 6    |      100.0%
goi_cuon           |    6 / 6    |      100.0%
mien_ga            |    6 / 6    |      100.0%
nem_ran            |    6 / 6    |      100.0%
pepsi              |    3 / 6    |       50.0%
pho                |    6 / 6    |      100.0%
tra_da             |    6 / 6    |      100.0%
unknown            |   15 / 16   |       93.8%
------------------------------------------------
TỔNG QUÁT (ONNX)   |   72 / 76   |       94.7%


In [33]:
import shutil
from google.colab import files

# 1. Lưu bản sao lưu vĩnh viễn lên Google Drive
shutil.copytree("outputs", "/content/drive/MyDrive/canteen_ai/outputs_v1", dirs_exist_ok=True)

# 2. Tải trực tiếp 2 file chạy thực tế về máy tính
files.download("../public/model/canteen_rush.onnx")
files.download("../public/model/labels.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>